# PaySim fraud detection: 01 data and exploration

**Goal:** predict which mobile money transactions are fraud.

**Data:** [PaySim](https://www.kaggle.com/datasets/ealaxi/paysim1), a simulated mobile money dataset: 6,362,620 transactions over ~31 days (`step` = hour of the simulation). Download the CSV from Kaggle, rename it `paysim.csv` and put it next to this notebook.

**This notebook:**
1. Data quality
2. Exploration: how does fraud behave?
3. Baseline: the bank's existing rule

Features and models are in `02_features_and_models.ipynb`.

## Summary of findings
1. **Fraud is very rare: 0.13%** (8,213 of 6.36M), so accuracy is the wrong metric. I use precision and recall.
2. **Fraud only happens in TRANSFER and CASH_OUT.** The model only needs these two types.
3. **Frauds are larger:** median 441k vs 75k for normal transactions (~6×). But the largest normal transaction (92M) is far bigger than any fraud, so amount alone isn't enough.
4. **Frauds are capped at 10M:** 287 frauds are exactly 10,000,000, which looks like a per-transaction limit.
5. **Frauds come in pairs:** a TRANSFER and a CASH_OUT of the same amount in the same hour.
6. **Fraud empties the account:** 98% of frauds leave the victim's balance at 0; most of the rest hit the 10M limit.
7. **Fraud is most common at night** (hours 0 to 6), when there are few transactions.
8. **The bank's rule (`isFlaggedFraud`) has 100% precision but only 0.2% recall:** it catches 16 of 8,213 frauds.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# show big numbers as 74,685 instead of 7.468500e+04
pd.set_option("display.float_format", "{:,.2f}".format)

# 1. Data quality

### Is the data clean? (size, types, missing values, duplicates)

In [ ]:
df = pd.read_csv("paysim.csv")
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
df.isna().sum()

In [ ]:
df.duplicated().sum()

The original column name `oldbalanceOrg` is spelled differently from `newbalanceOrig`, so I rename it to keep them consistent.

In [ ]:
df = df.rename(columns={"oldbalanceOrg": "oldbalanceOrig"})

### Are there negative money values?

In [ ]:
money_cols = ["amount", "oldbalanceOrig", "newbalanceOrig", "oldbalanceDest", "newbalanceDest"]
(df[money_cols] < 0).sum()

6,362,620 rows and 11 columns, with **no missing values, no duplicates and no negative amounts or balances**. The data needs no cleaning before exploration.

# 2. Exploration: how does fraud behave?

### 2a. How rare is fraud?

In [ ]:
df["isFraud"].value_counts()

In [ ]:
perc_of_fraud = df["isFraud"].mean() * 100
round(perc_of_fraud, 3)

In [ ]:
df["isFraud"].value_counts().plot(kind="bar", logy=True)
plt.title("Fraud vs normal transactions (log scale)")
plt.show()

Only **0.13%** of transactions are fraud (about 1 in 775).
A model that always predicts "not fraud" would be 99.87% accurate and catch nothing, so I judge models on **precision and recall**, not accuracy.

### 2b. Which transaction types have fraud?

In [ ]:
df["type"].value_counts()

In [ ]:
df.groupby("type")["isFraud"].mean() * 100

Fraud only happens in **TRANSFER** (0.77% of transfers) and **CASH_OUT** (0.18%). PAYMENT, DEBIT and CASH_IN have no fraud at all, so from here I keep only these two types in a table called `tc`.

In [ ]:
tc = df[df["type"].isin(["TRANSFER", "CASH_OUT"])].copy()
tc.shape

### 2c. Are fraud amounts different?

In [ ]:
df["amount"].describe()

The **mean (~180k) is more than double the median (~75k)**: a few very large transactions pull the average up, so the median is the fairer "typical" amount.

In [ ]:
df.groupby("isFraud")["amount"].describe()

Amounts are very skewed, so I plot them on a log scale to see both groups clearly.

In [ ]:
tc["log_amount"] = np.log1p(tc["amount"])

tc.groupby("isFraud")["log_amount"].plot(kind="hist", bins=50, alpha=0.5, density=True, legend=True)
plt.xlabel("log(amount + 1)")
plt.title("Amount: fraud vs normal (TRANSFER and CASH_OUT)")
plt.show()

- A typical fraud is about **6× bigger** than a normal transaction (median 441,423 vs 74,685).
- The largest fraud is **exactly 10,000,000**, which is suspiciously round.
- The largest *normal* transaction (92M) is much bigger than any fraud, and the two histograms overlap a lot, so amount helps but can't be the only signal.

### 2d. What do fraud amounts look like up close?

In [ ]:
fraud = df[df["isFraud"] == 1].copy()
len(fraud)

In [ ]:
fraud["amount"].value_counts().head()

- **287 frauds are exactly 10,000,000**, while the next most common amount appears only 4 times. This looks like a **per-transaction limit**.
- **16 frauds have an amount of 0**, which isn't realistic, so I note them as a simulation glitch.
- There are 8,213 frauds but only ~3,977 different amounts, so most amounts appear **twice**.

### 2e. Do frauds come in pairs?

In [ ]:
fraud[fraud["amount"] == 429257.45]

**Frauds come in pairs:** a TRANSFER and then a CASH_OUT of the same amount in the same hour. The money is moved out of the victim's account, then withdrawn.
The data doesn't link the two accounts directly, so this is inferred from amount and timing.

### 2f. Does fraud empty the account?

In [ ]:
fraud["emptied"] = fraud["newbalanceOrig"] == 0
fraud["emptied"].mean() * 100

In [ ]:
fraud.groupby("type")["emptied"].mean() * 100

In [ ]:
not_emptied = fraud[fraud["newbalanceOrig"] != 0]
print(len(not_emptied))
(not_emptied["amount"] == 10000000).mean() * 100

**98% of frauds leave the victim's balance at 0** (99.98% of fraud CASH_OUTs, 96% of fraud TRANSFERs). Of the 160 that don't, 91% are exactly 10M, meaning the account held more than the limit.
So almost every fraud tries to take the full balance, and **"account emptied" should be a strong feature** for the model.

### 2g. When does fraud happen?

`step` counts hours from the start of the simulation, so `step % 24` gives the hour of the day.

In [ ]:
tc["hour"] = tc["step"] % 24
fraud_by_hour = tc.groupby("hour")["isFraud"].mean() * 100
tx_hour = tc.groupby("hour").size()

In [ ]:
ax = tx_hour.plot(kind="bar", color="lightgrey")
ax.set_ylabel("Number of transactions")

ax2 = ax.twinx()
fraud_by_hour.plot(ax=ax2, color="red", marker="o")
ax2.set_ylabel("Fraud rate (%)")

plt.title("Transactions vs fraud rate by hour")
plt.show()

Very few transactions happen at night (hours 0 to 6), but the **fraud rate is highest then**. Fraudsters seem to keep a steady pace while normal customers are asleep, so `hour` is worth adding as a feature.

### 2h. Which numeric columns are linked to fraud?

In [ ]:
num_cols = ["amount", "oldbalanceOrig", "newbalanceOrig",
            "oldbalanceDest", "newbalanceDest", "hour", "isFraud"]
tc[num_cols].corr()["isFraud"].sort_values()

In [ ]:
tc.groupby("isFraud")[num_cols].median()

In [ ]:
tc.boxplot(column="oldbalanceDest", by="isFraud", showfliers=False)
plt.show()

- `oldbalanceOrig` has the strongest link with fraud (0.35): fraud targets accounts with money in them and drains them.
- All other correlations are weak. Correlation only measures straight-line relationships, so a weak number doesn't mean a column is useless (e.g. `emptied` and `hour` matter, as shown above).
- Destination balances are often lower for fraud, which matches money being sent to new "mule" accounts.

# 3. Baseline: how good is the bank's existing rule?

PaySim already has a rule, `isFlaggedFraud`. Any model has to beat it.

In [ ]:
cross = pd.crosstab(df["isFraud"], df["isFlaggedFraud"])
cross

In [ ]:
caught = cross.loc[1, 1]
all_frauds = cross.loc[1].sum()
all_flagged = cross[1].sum()

recall = caught / all_frauds * 100
precision = caught / all_flagged * 100

print("Recall:", round(recall, 2), "%")
print("Precision:", round(precision, 2), "%")

- **Precision 100%:** every transaction the rule flagged was really fraud.
- **Recall 0.2%:** it caught only 16 of 8,213 frauds and missed 8,197.

The rule never raises a false alarm, but it misses almost all fraud. A useful model must catch far more fraud while keeping false alarms reasonable.

## What this means for the model
- Keep only TRANSFER and CASH_OUT
- Build features from the findings: transaction type, amount, "account emptied", balance mismatches, empty destination accounts, hour
- Beat the bank's rule: catch far more than 0.2% of fraud while keeping false alarms low

Next: `02_features_and_models.ipynb`